In [13]:
# entity class
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class PrepareBaseModelConfig:
    root_dir: Path
    base_model_path: Path
    updated_base_model_path: Path
    param_image_size: list
    param_learning_rate: float
    param_including_top: bool
    param_weights: str
    param_classes: int

In [14]:
from pathlib import Path

from kidney_disease_prediction.constants import *
from kidney_disease_prediction.utils.common import read_yaml, create_directories
from kidney_disease_prediction.entity.config_entity import DataIngestionConfig
from kidney_disease_prediction import logger


class ConfigurationManager:
    def __init__(
        self,
        config_file_path: Path = CONFIG_FILE_PATH,
        params_file_path: Path = PARAMS_FILE_PATH
    ):
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)

        # Create root artifacts directory
        # create_directories([self.config.artifacts_root])

        logger.info("Configuration and params loaded successfully")

    def get_prepare_base_model_config(self) -> PrepareBaseModelConfig:
        config = self.config.prepare_base_model

        # Create prepare base model directory
        create_directories([config.root_dir])

        # Basic validation
        if not config.source_URL:
            raise ValueError(f"in config. yaml prepare_base_model.source_URLis empty.")

        return PrepareBaseModelConfig(
            root_dir=Path(config.root_dir),
            base_model_path=Path(config.base_model_path),
            updated_base_model_path=Path(config.updated_base_model_path),
            param_image_size=self.params.IMAGE_SIZE,
            param_learning_rate=self.params.LEARNING_RATE,
            param_including_top=self.params.INCLUDING_TOP,
            param_weights=self.params.WEIGHTS,
            param_classes=self.params.CLASSES
        )

In [15]:
import torch
import torch.nn as nn
from torchvision import models

from kidney_disease_prediction import logger


class PrepareBaseModel:
    def __init__(self, config: PrepareBaseModelConfig):
        self.config = config
        self.model = None

    def get_base_model(self):
        try:
            logger.info("Loading VGG16 model")

            # Load pretrained VGG16
            self.model = models.vgg16(pretrained=True)

            logger.info("Base VGG16 loaded")

            # Replace classifier if needed
            if not self.config.INCLUDING_TOP:
                in_features = self.model.classifier[6].in_features

                self.model.classifier[6] = nn.Linear(
                    in_features, self.config.CLASSES
                )

                logger.info(f"Modified classifier for {self.config.CLASSES} classes")

            # Freeze feature layers
            for param in self.model.features.parameters():
                param.requires_grad = False

            logger.info("Feature layers frozen")

            # Save model weights
            torch.save(self.model.state_dict(), self.config.base_model_path)

            logger.info(f"Model saved at {self.config.base_model_path}")

        except Exception as e:
            logger.exception(e)
            raise

In [ ]:
logger.info("Starting prepare base model stage")

try:
    config = ConfigurationManager()

    prepare_base_model_config = config.get_prepare_base_model_config()

    # prepare_base_model = PrepareBaseModel(config=prepare_base_model_config)

    # prepare_base_model.get_base_model()

    logger.info("Prepare base model stage completed successfully")

except Exception as e:
    logger.exception(e)
    raise

[2026-05-04 18:36:31,415: INFO: 1202788349 : Starting prepare base model stage]
[2026-05-04 18:36:31,416: ERROR: 1202788349 : [Errno 2] No such file or directory: 'config/config.yaml']
Traceback (most recent call last):
  File "/tmp/ipykernel_24891/1202788349.py", line 4, in <module>
    config = ConfigurationManager()
  File "/tmp/ipykernel_24891/1965156207.py", line 15, in __init__
    self.config = read_yaml(config_file_path)
  File "/home/shashank-saraswat/miniconda3/envs/kidney_env/lib/python3.10/site-packages/ensure/main.py", line 849, in __call__
    return_val = self.f(*args, **kwargs)
  File "/home/shashank-saraswat/Projects/Kidney-Disease-Classification/src/kidney_disease_prediction/utils/common.py", line 37, in read_yaml
    raise e
  File "/home/shashank-saraswat/Projects/Kidney-Disease-Classification/src/kidney_disease_prediction/utils/common.py", line 30, in read_yaml
    with open(path_to_yaml) as yaml_file:
FileNotFoundError: [Errno 2] No such file or directory: 'config

FileNotFoundError: [Errno 2] No such file or directory: 'config/config.yaml'